## Which products have the highest Revenue


In [0]:
%sql
SELECT 
    product_id, 
    product_name, 
    SUM(revenue) AS product_revenue, 
    SUM(quantity) AS unit_sold
FROM eshopdb.gold.fact_order_items
GROUP BY 
    product_id, product_name
ORDER BY product_revenue DESC
LIMIT 10


## Which category drive the most revenue? 



In [0]:
%sql
SELECT  
    product_category, 
    SUM(revenue) AS category_revenue, 
    SUM(quantity) AS unit_sold
FROM eshopdb.gold.fact_order_items
GROUP BY product_category
ORDER BY category_revenue DESC

## What is the Average Order Value (AOV) By month?


In [0]:
%sql
SELECT 
    MONTH(order_date) AS month,
    SUM(revenue) AS total_revenue, 
    COUNT(DISTINCT order_id) AS units_sold, 
    SUM(revenue) / COUNT(DISTINCT customer_id) AS avg_order_value
FROM eshopdb.gold.fact_order_items
GROUP BY month
ORDER BY avg_order_value desc
LIMIT 10;

## Which countries generate the most revenue?

In [0]:
%sql
SELECT 
    customer_country as country, 
    SUM(o.revenue)  AS total_revenue, 
    COUNT(order_id) AS order_count
FROM eshopdb.gold.fact_orders o
GROUP BY customer_country
ORDER BY total_revenue DESC
limit 10;

## Who are the top customers by revenue?


In [0]:
%sql
SELECT 
    o.customer_id, 
    MAX(c.email) AS email, 
    MAX(c.country) AS country, 
    SUM(o.revenue)  AS total_revenue, 
    COUNT(order_id) AS order_count
FROM eshopdb.gold.fact_orderS o
LEFT JOIN eshopdb.gold.scd_customers c
    ON o.customer_id = c.customer_id
    AND c.dbt_valid_to is NULL
GROUP BY o.customer_id
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT * 
FROM   eshopdb.gold.fact_orders
LIMIT 5;

## Who are the customers who have more than one order (repeat customers)?

In [0]:
%sql
WITH orders_by_customer AS 
(
    SELECT 
    customer_id, 
    COUNT(order_id) AS order_count
    FROM eshopdb.gold.fact_orders
    WHERE customer_id IS NOT NULL AND 
        order_id IS NOT NULL 
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
), 
repeat_customers AS (
    SELECT 
        c.customer_id, 
        c.email, 
        c.country,
        oc.order_count AS no_of_orders
    FROM orders_by_customer oc
    JOIN eshopdb.gold.scd_customers c
        ON oc.customer_id = c.customer_id
        AND c.dbt_valid_to is NULL
)
SELECT * FROM repeat_customers;
